In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

# -----------------------------
# Experiment settings
# -----------------------------
random_state = 42

minority_samples = 1000

imbalance_ratios = [
    10, 100, 500, 1000, 5000, 10000, 20000,
]

n_splits = 5

rf_results = []
gb_results = []

cv = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=random_state
)

# -----------------------------
# Run experiment
# -----------------------------
for ir in imbalance_ratios:

    majority_samples = minority_samples * ir
    total_samples = majority_samples + minority_samples

    weights = [
        majority_samples / total_samples,
        minority_samples / total_samples,
    ]

    X, y = make_classification(
        n_samples=total_samples,
        n_features=20,
        n_informative=10,
        n_redundant=5,
        n_repeated=0,
        n_classes=2,
        weights=weights,
        flip_y=0,
        class_sep=0.7,
        random_state=random_state,
    )

    rf = RandomForestClassifier(
        n_estimators=200,
        random_state=random_state,
        n_jobs=-1
    )

    gb = GradientBoostingClassifier(
        random_state=random_state
    )

    rf_scores = cross_val_score(
        rf,
        X,
        y,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )

    gb_scores = cross_val_score(
        gb,
        X,
        y,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )

    rf_results.append((rf_scores.mean(), rf_scores.std()))
    gb_results.append((gb_scores.mean(), gb_scores.std()))

    print(
        f"IR {ir:4d}:1 | "
        f"RF = {rf_scores.mean():.4f} ± {rf_scores.std():.4f} | "
        f"GB = {gb_scores.mean():.4f} ± {gb_scores.std():.4f}"
    )

# -----------------------------
# Prepare results
# -----------------------------
rf_mean = [r[0] for r in rf_results]
rf_std = [r[1] for r in rf_results]

gb_mean = [r[0] for r in gb_results]
gb_std = [r[1] for r in gb_results]

# -----------------------------
# Plot
# -----------------------------
plt.figure(figsize=(8, 5))

plt.errorbar(
    imbalance_ratios,
    rf_mean,
    yerr=rf_std,
    marker="o",
    capsize=4,
    label="Random Forest"
)

plt.errorbar(
    imbalance_ratios,
    gb_mean,
    yerr=gb_std,
    marker="s",
    capsize=4,
    label="Gradient Boosting"
)

plt.xscale("log")
plt.xticks(imbalance_ratios, imbalance_ratios)

plt.xlabel("Imbalance Ratio (Majority : Minority)")
plt.ylabel("ROC-AUC")
plt.title("ROC-AUC vs Imbalance Ratio")
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

IR   10:1 | RF = 0.9746 ± 0.0062 | GB = 0.9587 ± 0.0072
IR  100:1 | RF = 0.9512 ± 0.0101 | GB = 0.9105 ± 0.0120
IR  500:1 | RF = 0.9389 ± 0.0080 | GB = 0.8867 ± 0.0123
IR 1000:1 | RF = 0.9267 ± 0.0088 | GB = 0.8565 ± 0.0274
IR 5000:1 | RF = 0.8650 ± 0.0109 | GB = 0.7987 ± 0.0064


Python(4504) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(4505) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(4506) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(4507) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(4508) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(4509) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
